## Loading Packages

In [34]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium

from tqdm import tqdm
from pathlib import Path

## Demo Trip DF


In [35]:
trip_demo = pd.DataFrame({
    "ride_id": list(range(1, 21)),

    "start_station": [
        "Station A", "Station A", "Station A", "Station A", "Station A",
        "Station B", "Station B", "Station B", "Station B",
        "Station C", "Station C", "Station C",
        "Station D", "Station D", "Station D",
        "Station E", "Station E",
        "Station A", "Station B", "Station C"
    ],

    "end_station": [
        "Station B", "Station B", "Station B", "Station C", "Station C",
        "Station A", "Station A", "Station C", "Station D",
        "Station A", "Station B", "Station E",
        "Station A", "Station C", "Station E",
        "Station A", "Station D",
        "Station E", "Station E", "Station D"
    ],

    "started_at": pd.to_datetime([
        "2025-01-01 08:00", "2025-01-01 08:15", "2025-01-01 08:30",
        "2025-01-01 09:00", "2025-01-01 09:20",
        "2025-01-01 10:00", "2025-01-01 10:15", "2025-01-01 10:40",
        "2025-01-01 11:00",
        "2025-01-01 11:30", "2025-01-01 12:00", "2025-01-01 12:20",
        "2025-01-01 13:00", "2025-01-01 13:30", "2025-01-01 14:00",
        "2025-01-01 14:30", "2025-01-01 15:00",
        "2025-01-01 15:30", "2025-01-01 16:00", "2025-01-01 16:30"
    ]),

    "ended_at": pd.to_datetime([
        "2025-01-01 08:10", "2025-01-01 08:28", "2025-01-01 08:42",
        "2025-01-01 09:18", "2025-01-01 09:35",
        "2025-01-01 10:12", "2025-01-01 10:30", "2025-01-01 10:55",
        "2025-01-01 11:18",
        "2025-01-01 11:48", "2025-01-01 12:14", "2025-01-01 12:45",
        "2025-01-01 13:20", "2025-01-01 13:48", "2025-01-01 14:20",
        "2025-01-01 14:55", "2025-01-01 15:22",
        "2025-01-01 15:58", "2025-01-01 16:25", "2025-01-01 16:50"
    ]),

    "member_casual": [
        "member", "member", "casual", "member", "casual",
        "member", "member", "casual", "member",
        "casual", "member", "casual",
        "member", "casual", "member",
        "casual", "member",
        "member", "casual", "member"
    ]
})

trip_demo

,ride_id,start_station,end_station,started_at,ended_at,member_casual
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual
5,6,Station B,Station A,2025-01-01 10:00:00,2025-01-01 10:12:00,member
6,7,Station B,Station A,2025-01-01 10:15:00,2025-01-01 10:30:00,member
7,8,Station B,Station C,2025-01-01 10:40:00,2025-01-01 10:55:00,casual
8,9,Station B,Station D,2025-01-01 11:00:00,2025-01-01 11:18:00,member
9,10,Station C,Station A,2025-01-01 11:30:00,2025-01-01 11:48:00,casual


In [36]:
station_coordinates = pd.DataFrame({
    "station": [
        "Station A",
        "Station B",
        "Station C",
        "Station D",
        "Station E"
    ],
    "lat": [
        40.735,
        40.751,
        40.742,
        40.728,
        40.760
    ],
    "lng": [
        -73.991,
        -73.977,
        -73.985,
        -73.970,
        -73.995
    ]
})

station_coordinates

,station,lat,lng
0,Station A,40.735,-73.991
1,Station B,40.751,-73.977
2,Station C,40.742,-73.985
3,Station D,40.728,-73.970
4,Station E,40.760,-73.995


In [37]:
start_df = trip_demo.merge(station_coordinates, how='left', left_on='start_station', right_on='station')
start_df.rename(columns={
    'lat': 'start_lat',
    'lng': 'start_lng'
}, inplace=True)

In [38]:
start_df.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,station,start_lat,start_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,Station A,40.735,-73.991
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,Station A,40.735,-73.991
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,Station A,40.735,-73.991
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,Station A,40.735,-73.991
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,Station A,40.735,-73.991


In [39]:
end_df = trip_demo.merge(station_coordinates, how='left', left_on='end_station', right_on='station')
end_df.rename(columns={
    'lat': 'end_lat',
    'lng': 'end_lng'
}, inplace=True)

end_df.drop(columns=['station'], inplace=True)

end_df.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,end_lat,end_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.751,-73.977
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.751,-73.977
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.751,-73.977
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.742,-73.985
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.742,-73.985


In [40]:
start_df.columns
start_cols = ['ride_id', 'start_station', 'started_at', 'start_lat', 'start_lng']
end_cols = ['ride_id', 'end_station', 'ended_at', 'end_lat', 'end_lng', 'member_casual']

In [41]:
final_df = pd.merge(start_df[start_cols], end_df[end_cols], how='inner', on='ride_id')
final_df.head() 

,ride_id,start_station,started_at,start_lat,start_lng,end_station,ended_at,end_lat,end_lng,member_casual
0,1,Station A,2025-01-01 08:00:00,40.735,-73.991,Station B,2025-01-01 08:10:00,40.751,-73.977,member
1,2,Station A,2025-01-01 08:15:00,40.735,-73.991,Station B,2025-01-01 08:28:00,40.751,-73.977,member
2,3,Station A,2025-01-01 08:30:00,40.735,-73.991,Station B,2025-01-01 08:42:00,40.751,-73.977,casual
3,4,Station A,2025-01-01 09:00:00,40.735,-73.991,Station C,2025-01-01 09:18:00,40.742,-73.985,member
4,5,Station A,2025-01-01 09:20:00,40.735,-73.991,Station C,2025-01-01 09:35:00,40.742,-73.985,casual


### Map Center

In [42]:
map_center = [
    pd.concat([final_df['start_lat'], final_df['end_lat']]).mean(),
    pd.concat([final_df['start_lng'], final_df['end_lng']]).mean()
]

map_center

[np.float64(40.7427), np.float64(-73.9841)]

### Adding Duration

In [43]:
final_df["duration_min"] = (
    final_df['ended_at'] -final_df['started_at']
    ).dt.total_seconds() / 60 

final_df[[
    "ride_id",
    "start_station",
    "end_station",
    "started_at",
    "ended_at",
    "duration_min"
]].head()

,ride_id,start_station,end_station,started_at,ended_at,duration_min
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,10.0
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,13.0
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,12.0
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,18.0
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,15.0


### Adding Feature Time

In [44]:
trip_demo["date"] = trip_demo["started_at"].dt.date
trip_demo["hour"] = trip_demo["started_at"].dt.hour
trip_demo["day_name"] = trip_demo["started_at"].dt.day_name()
trip_demo["month_name"] = trip_demo["started_at"].dt.month_name()

trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,date,hour,day_name,month_name
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,2025-01-01,8,Wednesday,January
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,2025-01-01,8,Wednesday,January
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,2025-01-01,8,Wednesday,January
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,2025-01-01,9,Wednesday,January
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,2025-01-01,9,Wednesday,January


### Point

In [45]:
map_center

[np.float64(40.7427), np.float64(-73.9841)]

In [46]:
m = folium.Map(
    location=map_center,
    zoom_start=13
)

for _, row in station_coordinates.iterrows():
    folium.Marker(
        location=[row['lat'], row['lng']],
        popup=row['station']
    ).add_to(m)
    
m

### Line 

In [47]:
start_point = [final_df.loc[0, 'start_lat'], final_df.loc[0, 'start_lng']]
end_point = [final_df.loc[0, 'end_lat'], final_df.loc[0, 'end_lng']]

folium.Marker(start_point, popup='Start').add_to(m)
folium.Marker(end_point, popup='End').add_to(m)

folium.PolyLine(
    locations=[start_point, end_point],
    weight=5,
    opacity=0.8
).add_to(m)

m

In [48]:
print(final_df.columns)

Index(['ride_id', 'start_station', 'started_at', 'start_lat', 'start_lng',
       'end_station', 'ended_at', 'end_lat', 'end_lng', 'member_casual',
       'duration_min'],
      dtype='str')


### Flow Data

In [49]:
flow_data = (
    final_df
    .groupby(['start_station', 'end_station'],
             as_index=False
    )
    .agg(
        number_of_rides=('ride_id', 'count'),
        avg_duration_min=('duration_min', 'mean'),
        start_lat=('start_lat', 'mean'),
        start_lng=('start_lng', 'mean'),
        end_lat=('end_lat', 'mean'),
        end_lng=('end_lng', 'mean')
    )
    .sort_values('number_of_rides', ascending=False)
)

flow_data

,start_station,end_station,number_of_rides,avg_duration_min,start_lat,start_lng,end_lat,end_lng
0,Station A,Station B,3,11.666667,40.735,-73.991,40.751,-73.977
1,Station A,Station C,2,16.500000,40.735,-73.991,40.742,-73.985
3,Station B,Station A,2,13.500000,40.751,-73.977,40.735,-73.991
2,Station A,Station E,1,28.000000,40.735,-73.991,40.760,-73.995
4,Station B,Station C,1,15.000000,40.751,-73.977,40.742,-73.985
5,Station B,Station D,1,18.000000,40.751,-73.977,40.728,-73.970
6,Station B,Station E,1,25.000000,40.751,-73.977,40.760,-73.995
7,Station C,Station A,1,18.000000,40.742,-73.985,40.735,-73.991
8,Station C,Station B,1,14.000000,40.742,-73.985,40.751,-73.977
9,Station C,Station D,1,20.000000,40.742,-73.985,40.728,-73.970


#### Visualizing Flows


In [50]:
import branca.colormap as cm

linear_cmap = cm.LinearColormap(
    colors=['blue', 'green', 'yellow', 'orange', 'red'],
    vmin=flow_data['number_of_rides'].min(),
    vmax=flow_data['number_of_rides'].max()
)

max_rides = flow_data['number_of_rides'].max()

for _, row in flow_data.iterrows():
    start_point = [row['start_lat'], row['start_lng']]
    end_point = [row['end_lat'], row['end_lng']]
    
flow_map = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles='CartoDB positron'
)

for _, row in station_coordinates.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=6,
        popup=row['station'],
        tooltip=row['station'],
        fill=True
    ).add_to(flow_map)
    
    max_rides = flow_data['number_of_rides'].max()
    
    for _, row in flow_data.iterrows():
        
        start_point = [row['start_lat'], row['start_lng']]
        end_point = [row['end_lat'], row['end_lng']]
        
        line_width = 1 + 7 * row['number_of_rides'] / max_rides
        
        line_color = linear_cmap(row['number_of_rides'])

        
        popup_text = (
             f"<b>{row['start_station']} → {row['end_station']}</b><br>"
            f"Number of rides: {row['number_of_rides']}<br>"
            f"Average duration: {row['avg_duration_min']:.1f} minutes"
        )
        
        folium.PolyLine(
            locations=[start_point, end_point],
            weight=line_width,
            color=line_color,
            opacity=0.7,
            popup=popup_text,
            tooltip=f"{row['start_station']} →{row['end_station']}"
        ).add_to(flow_map)
        
flow_map

### Bounding Box


In [51]:
min_lat = min(final_df['start_lat'].min(), final_df['end_lat'].min())
max_lat = max(final_df["start_lat"].max(), final_df["end_lat"].max())

min_lng = min(final_df["start_lng"].min(), final_df["end_lng"].min())
max_lng = max(final_df["start_lng"].max(), final_df["end_lng"].max())

bounding_box = pd.DataFrame({
    'metric': ['min_lat', 'max_lat', 'min_lng', 'max_lng'],
    'value': [min_lat, max_lat, min_lng, max_lng]
})

bounding_box

,metric,value
0,min_lat,40.728
1,max_lat,40.760
2,min_lng,-73.995
3,max_lng,-73.970


In [52]:
bbox_map = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles="CartoDB positron"
)

for _, row in station_coordinates.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=6,
        popup=row['station'],
        tooltip=row['station'],
        fill=True
    ).add_to(bbox_map)
    
    bbox_coordinates = [
        [min_lat, min_lng],
        [min_lat, max_lng],
        [max_lat, max_lng],
        [max_lat, min_lng],
        [min_lat, min_lng]
    ]
    
    folium.PolyLine(
    locations=bbox_coordinates,
    weight=4,
    opacity=0.8,
    popup="Bounding Box"
).add_to(bbox_map)

bbox_map        
    

### Distance

In [53]:
import geopandas as gpd
start_points = gpd.GeoDataFrame(
    final_df.copy(),
    geometry=gpd.points_from_xy(
        final_df['start_lng'],
        final_df['start_lat']
    ),
    crs='EPSG:4326'
)

end_points = gpd.GeoDataFrame(
    final_df.copy(),
    geometry=gpd.points_from_xy(
        final_df['end_lng'],
        final_df['end_lat']
    ),
    crs="EPSG:4326"
)

In [54]:
start_points[[
    "ride_id",
    "start_station",
    "end_station",
    "geometry"
]].head()

,ride_id,start_station,end_station,geometry
0,1,Station A,Station B,POINT (-73.991 40.735)
1,2,Station A,Station B,POINT (-73.991 40.735)
2,3,Station A,Station B,POINT (-73.991 40.735)
3,4,Station A,Station C,POINT (-73.991 40.735)
4,5,Station A,Station C,POINT (-73.991 40.735)


In [55]:
projected_crs = "EPSG:32618"

start_points_projected = start_points.to_crs(projected_crs)
end_points_projected = end_points.to_crs(projected_crs)

In [56]:
final_df['distance_m'] = start_points_projected.geometry.distance(
    end_points_projected.geometry
)

final_df['distance_km'] = final_df['distance_m'] / 1000

final_df.head()

,ride_id,start_station,started_at,start_lat,start_lng,end_station,ended_at,end_lat,end_lng,member_casual,duration_min,distance_m,distance_km
0,1,Station A,2025-01-01 08:00:00,40.735,-73.991,Station B,2025-01-01 08:10:00,40.751,-73.977,member,10.0,2133.621698,2.133622
1,2,Station A,2025-01-01 08:15:00,40.735,-73.991,Station B,2025-01-01 08:28:00,40.751,-73.977,member,13.0,2133.621698,2.133622
2,3,Station A,2025-01-01 08:30:00,40.735,-73.991,Station B,2025-01-01 08:42:00,40.751,-73.977,casual,12.0,2133.621698,2.133622
3,4,Station A,2025-01-01 09:00:00,40.735,-73.991,Station C,2025-01-01 09:18:00,40.742,-73.985,member,18.0,927.671157,0.927671
4,5,Station A,2025-01-01 09:20:00,40.735,-73.991,Station C,2025-01-01 09:35:00,40.742,-73.985,casual,15.0,927.671157,0.927671


## Jersey

In [57]:
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    
    year list of strings
    """
    YEAR =year
    MONTH = [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]
    
    periods = []
    
    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
        return periods

In [58]:
PERIODS = period_iterator(["2025"],0,12)
PERIODS

['202501',
 '202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError

CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)


for i in PERIODS:

    try:
        file_name = f"JC-{i}-citibike-tripdata.csv.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    except (HTTPError, URLError, FileNotFoundError):
        file_name = f"JC-{i}-citibike-tripdata.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)
    print(f'{file_name}  Extracted')
    zip_path.unlink()
    print(f"{file_name} removed.")


JC-202501-citibike-tripdata.csv.zip  Extracted
JC-202501-citibike-tripdata.csv.zip removed.
JC-202502-citibike-tripdata.csv.zip  Extracted
JC-202502-citibike-tripdata.csv.zip removed.
JC-202503-citibike-tripdata.csv.zip  Extracted
JC-202503-citibike-tripdata.csv.zip removed.
JC-202504-citibike-tripdata.csv.zip  Extracted
JC-202504-citibike-tripdata.csv.zip removed.
JC-202505-citibike-tripdata.csv.zip  Extracted
JC-202505-citibike-tripdata.csv.zip removed.
JC-202506-citibike-tripdata.csv.zip  Extracted
JC-202506-citibike-tripdata.csv.zip removed.
JC-202507-citibike-tripdata.csv.zip  Extracted
JC-202507-citibike-tripdata.csv.zip removed.
JC-202508-citibike-tripdata.csv.zip  Extracted
JC-202508-citibike-tripdata.csv.zip removed.
JC-202509-citibike-tripdata.csv.zip  Extracted
JC-202509-citibike-tripdata.csv.zip removed.
